In [2]:
w1
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import csv

In [2]:

df = pd.read_csv("pune_mumbai_combined.csv")

In [3]:

df.shape 

(600, 13)

In [7]:
df.columns.tolist()    

['Locality',
 'Title',
 'Price_Rs',
 'Furnishing',
 'Bathrooms',
 'Carpet_Area_sqft',
 'Floor_No',
 'BHK',
 'Property_Type',
 'Listing_Type',
 'City',
 'Society',
 'Price_per_sqft']

In [9]:
df.head(5)    # first 5 rows

,Locality,Title,Price_Rs,Furnishing,Bathrooms,Carpet_Area_sqft,Floor_No,BHK,Property_Type,Listing_Type,City,Society,Price_per_sqft
0,Wakad,"3 BHK Flat for Rent in Wakad, Pune",60000.0,Furnished,3.0,1086.0,7.0,3,Flat,Rent,NaN,NaN,55.25
1,Wakad,"2 BHK Flat for Rent in Ganesh Imperia, Wakad, ...",19500.0,Semi-Furnished,2.0,NaN,2.0,2,Flat,Rent,NaN,NaN,NaN
2,Wakad,"2 BHK Flat for Rent in Paranjape Broadway, Wak...",27000.0,Furnished,2.0,NaN,5.0,2,Flat,Rent,NaN,NaN,NaN
3,Wakad,"3 BHK Flat for Rent in Paramount Altissimo, Wa...",42000.0,Semi-Furnished,3.0,NaN,8.0,3,Flat,Rent,NaN,NaN,NaN
4,Wakad,"2 BHK Flat for Rent in Wakad, Pune",38000.0,Furnished,2.0,NaN,11.0,2,Flat,Rent,NaN,NaN,NaN


In [13]:
df.tail(5)    # last 5 rows  

,Locality,Title,Price_Rs,Furnishing,Bathrooms,Carpet_Area_sqft,Floor_No,BHK,Property_Type,Listing_Type,City,Society,Price_per_sqft
295,Goregaon East,"5 BHK Flat for Rent in Oberoi Woods, Goregaon ...",3.0,Semi-Furnished,3.0,1900.0,19.0,5,Flat,Rent,Mumbai,Oberoi Woods,0.0
296,Goregaon East,"5 BHK Flat for Rent in Oberoi Woods, Goregaon ...",3.3,Semi-Furnished,4.0,1966.0,12.0,5,Flat,Rent,Mumbai,Oberoi Woods,0.0
297,Goregaon East,"3 BHK Flat for Rent in Oberoi Exquisite II, Go...",1.9,Semi-Furnished,3.0,1245.0,27.0,3,Flat,Rent,Mumbai,Oberoi Exquisite II,0.0
298,Goregaon East,"3 BHK Flat for Rent in Oberoi Exquisite, Goreg...",1.8,Semi-Furnished,3.0,1041.0,28.0,3,Flat,Rent,Mumbai,Oberoi Exquisite,0.0
299,Goregaon East,"4 BHK Flat for Rent in Meenaxi Apartments, Gor...",1.4,Furnished,4.0,1600.0,7.0,4,Flat,Rent,Mumbai,Meenaxi Apartments,0.0


In [11]:
df.info()           # dtypes + non-null counts 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Locality          600 non-null    object 
 1   Title             600 non-null    object 
 2   Price_Rs          600 non-null    float64
 3   Furnishing        599 non-null    object 
 4   Bathrooms         598 non-null    float64
 5   Carpet_Area_sqft  500 non-null    float64
 6   Floor_No          560 non-null    float64
 7   BHK               600 non-null    int64  
 8   Property_Type     600 non-null    object 
 9   Listing_Type      600 non-null    object 
 10  City              300 non-null    object 
 11  Society           266 non-null    object 
 12  Price_per_sqft    500 non-null    float64
dtypes: float64(5), int64(1), object(7)
memory usage: 61.1+ KB


In [27]:
df.isna().sum()  # where cleaning is needed 

Locality              0
Title                 0
Price_Rs              0
Furnishing            0
Bathrooms             0
Carpet_Area_sqft      0
Floor_No              0
BHK                   0
Property_Type         0
Listing_Type          0
City                300
Society             334
Price_per_sqft      100
dtype: int64

In [19]:
df.describe(include='all')      # stats summary 

,Locality,Title,Price_Rs,Furnishing,Bathrooms,Carpet_Area_sqft,Floor_No,BHK,Property_Type,Listing_Type,City,Society,Price_per_sqft
count,300,300,300.000000,300,299.000000,294.000000,265.000000,300.000000,300,300,300,266,294.000000
unique,10,214,NaN,3,NaN,NaN,NaN,NaN,1,1,1,158,NaN
top,Andheri West,"3 BHK Flat for Rent in Oberoi Sky City, Boriva...",NaN,Semi-Furnished,NaN,NaN,NaN,NaN,Flat,Rent,Mumbai,Oberoi Sky City,NaN
freq,30,11,NaN,160,NaN,NaN,NaN,NaN,300,300,300,14,NaN
mean,NaN,NaN,34564.231667,NaN,2.705686,1041.993197,15.992453,2.610000,NaN,NaN,NaN,NaN,46.437109
std,NaN,NaN,34923.297000,NaN,0.919769,480.143579,11.145726,0.852637,NaN,NaN,NaN,NaN,45.699898
min,NaN,NaN,1.000000,NaN,1.000000,290.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,0.000000
25%,NaN,NaN,1.800000,NaN,2.000000,740.250000,8.000000,2.000000,NaN,NaN,NaN,NaN,0.000000
50%,NaN,NaN,37000.000000,NaN,3.000000,992.500000,13.000000,3.000000,NaN,NaN,NaN,NaN,62.750000
75%,NaN,NaN,67250.000000,NaN,3.000000,1200.000000,22.000000,3.000000,NaN,NaN,NaN,NaN,86.632500


In [61]:
# Column Name Standardization

df.columns = [c.strip().replace(' ', '_') for c in df.columns]
df.columns

Index(['Locality', 'Title', 'Price_Rs', 'Furnishing', 'Bathrooms',
       'Carpet_Area_sqft', 'Floor_No', 'BHK', 'Property_Type', 'Listing_Type',
       'City', 'Society', 'Price_per_sqft'],
      dtype='object')

In [63]:
df['Furnishing'].fillna('Unknown', inplace=True)

In [65]:
df['Bathrooms'].fillna(df['Bathrooms'].median(), inplace=True)

In [67]:
df['Floor_No'].fillna(df['Floor_No'].median(), inplace=True)

In [69]:
df['Carpet_Area_sqft'] = df.groupby('Locality')['Carpet_Area_sqft']\
                            .transform(lambda x: x.fillna(x.median()))


In [71]:
df['Price_per_sqft'] = (df['Price_Rs'] / df['Carpet_Area_sqft']).round(2)

In [73]:
df = df[
    (df['Price_Rs'] >= 5000) &
    (df['Price_Rs'] <= 300000) &
    (df['Carpet_Area_sqft'] >= 200) &
    (df['Carpet_Area_sqft'] <= 4000)
]


In [75]:
df.duplicated().sum()

0

In [77]:
df.drop_duplicates(inplace=True)

In [79]:
df.isna().sum()

Locality              0
Title                 0
Price_Rs              0
Furnishing            0
Bathrooms             0
Carpet_Area_sqft      0
Floor_No              0
BHK                   0
Property_Type         0
Listing_Type          0
City                292
Society             314
Price_per_sqft        0
dtype: int64

In [81]:
df.describe()

,Price_Rs,Bathrooms,Carpet_Area_sqft,Floor_No,BHK,Price_per_sqft
count,451.000000,451.000000,451.000000,451.000000,451.000000,451.000000
mean,46909.478936,2.184035,883.698448,8.725055,2.152993,56.038071
std,21515.221793,0.606859,266.176163,7.260840,0.643676,27.649410
min,5000.000000,1.000000,250.000000,1.000000,1.000000,9.240000
25%,30500.000000,2.000000,731.000000,4.000000,2.000000,35.545000
50%,42000.000000,2.000000,860.000000,7.000000,2.000000,47.270000
75%,60000.000000,3.000000,1050.000000,11.000000,3.000000,76.965000
max,99000.000000,4.000000,3000.000000,50.000000,4.000000,152.630000


In [83]:
df.shape

(451, 13)

In [59]:
df['Locality'].value_counts().head(5)

Locality
Wakad        30
Balewadi     30
Hinjawadi    30
Wagholi      30
Kothrud      30
Name: count, dtype: int64

In [172]:
df.isna().sum()

City                0
Locality            0
Property_Type       0
BHK                 0
Carpet_Area_sqft    0
Floor_No            0
Bathrooms           0
Furnishing          0
Price_Rs            0
Price_per_sqft      0
Listing_Type        0
Title               0
dtype: int64

In [106]:
df[['City','Society']].isna().sum()

City       292
Society    314
dtype: int64

In [109]:
df.drop(columns=['Society'], inplace=True)


In [125]:
df

,Locality,Title,Price_Rs,Furnishing,Bathrooms,Carpet_Area_sqft,Floor_No,BHK,Property_Type,Listing_Type,City,Price_per_sqft
0,Wakad,"3 BHK Flat for Rent in Wakad, Pune",60000.0,Furnished,3.0,1086.0,7.0,3,Flat,Rent,NaN,55.25
1,Wakad,"2 BHK Flat for Rent in Ganesh Imperia, Wakad, ...",19500.0,Semi-Furnished,2.0,923.5,2.0,2,Flat,Rent,NaN,21.12
2,Wakad,"2 BHK Flat for Rent in Paranjape Broadway, Wak...",27000.0,Furnished,2.0,923.5,5.0,2,Flat,Rent,NaN,29.24
3,Wakad,"3 BHK Flat for Rent in Paramount Altissimo, Wa...",42000.0,Semi-Furnished,3.0,923.5,8.0,3,Flat,Rent,NaN,45.48
4,Wakad,"2 BHK Flat for Rent in Wakad, Pune",38000.0,Furnished,2.0,923.5,11.0,2,Flat,Rent,NaN,41.15
...,...,...,...,...,...,...,...,...,...,...,...,...
589,Goregaon East,"2 BHK Flat for Rent in Avant Hillway, Goregaon...",55000.0,Semi-Furnished,2.0,756.0,25.0,2,Flat,Rent,Mumbai,72.75
590,Goregaon East,"2 BHK Flat for Rent in suchidham complex, Gore...",54000.0,Furnished,2.0,550.0,3.0,2,Flat,Rent,Mumbai,98.18
591,Goregaon East,"2 BHK Flat for Rent in Goregaon East, Mumbai",70000.0,Unfurnished,2.0,757.0,33.0,2,Flat,Rent,Mumbai,92.47
592,Goregaon East,"2 BHK Flat for Rent in Satellite Garden, Goreg...",60000.0,Furnished,2.0,650.0,5.0,2,Flat,Rent,Mumbai,92.31


In [128]:
df.dtypes

Locality             object
Title                object
Price_Rs            float64
Furnishing           object
Bathrooms           float64
Carpet_Area_sqft    float64
Floor_No            float64
BHK                   int64
Property_Type        object
Listing_Type         object
City                 object
Price_per_sqft      float64
dtype: object

In [136]:
pune_localities = [
    'Wakad','Kharadi','Hinjawadi','Wagholi','Baner','Hadapsar',
    'Balewadi','Kothrud','Magarpatta City','Viman Nagar'
]

df['City'] = df['Locality'].apply(lambda x: 'Pune' if x in pune_localities else 'Mumbai')


In [138]:
df['City'].isna().sum()


0

In [152]:
# Arangement of columns 

ordered_cols = [
    'City',
    'Locality',
    'Property_Type',
    'BHK',
    'Carpet_Area_sqft',
    'Floor_No',
    'Bathrooms',
    'Furnishing',
    'Price_Rs',
    'Price_per_sqft',
    'Listing_Type',
    'Title'
]

df = df[ordered_cols]


In [162]:
df.to_csv('pune_mumbai_combined_clean_data.csv', index=False)

In [9]:
df = pd.read_csv("pune_mumbai_combined_clean_data.csv")

In [11]:
df

,City,Locality,Property_Type,BHK,Carpet_Area_sqft,Floor_No,Bathrooms,Furnishing,Price_Rs,Price_per_sqft,Listing_Type,Title
0,Pune,Wakad,Flat,3,1086.0,7.0,3.0,Furnished,60000.0,55.25,Rent,"3 BHK Flat for Rent in Wakad, Pune"
1,Pune,Wakad,Flat,2,923.5,2.0,2.0,Semi-Furnished,19500.0,21.12,Rent,"2 BHK Flat for Rent in Ganesh Imperia, Wakad, ..."
2,Pune,Wakad,Flat,2,923.5,5.0,2.0,Furnished,27000.0,29.24,Rent,"2 BHK Flat for Rent in Paranjape Broadway, Wak..."
3,Pune,Wakad,Flat,3,923.5,8.0,3.0,Semi-Furnished,42000.0,45.48,Rent,"3 BHK Flat for Rent in Paramount Altissimo, Wa..."
4,Pune,Wakad,Flat,2,923.5,11.0,2.0,Furnished,38000.0,41.15,Rent,"2 BHK Flat for Rent in Wakad, Pune"
...,...,...,...,...,...,...,...,...,...,...,...,...
446,Mumbai,Goregaon East,Flat,2,756.0,25.0,2.0,Semi-Furnished,55000.0,72.75,Rent,"2 BHK Flat for Rent in Avant Hillway, Goregaon..."
447,Mumbai,Goregaon East,Flat,2,550.0,3.0,2.0,Furnished,54000.0,98.18,Rent,"2 BHK Flat for Rent in suchidham complex, Gore..."
448,Mumbai,Goregaon East,Flat,2,757.0,33.0,2.0,Unfurnished,70000.0,92.47,Rent,"2 BHK Flat for Rent in Goregaon East, Mumbai"
449,Mumbai,Goregaon East,Flat,2,650.0,5.0,2.0,Furnished,60000.0,92.31,Rent,"2 BHK Flat for Rent in Satellite Garden, Goreg..."


In [27]:
df.head()

,City,Locality,Property_Type,BHK,Carpet_Area_sqft,Floor_No,Bathrooms,Furnishing,Price_Rs,Price_per_sqft,Listing_Type,Title
0,Pune,Wakad,Flat,3,1086.0,7.0,3.0,Furnished,60000.0,55.25,Rent,"3 BHK Flat for Rent in Wakad, Pune"
1,Pune,Wakad,Flat,2,923.5,2.0,2.0,Semi-Furnished,19500.0,21.12,Rent,"2 BHK Flat for Rent in Ganesh Imperia, Wakad, ..."
2,Pune,Wakad,Flat,2,923.5,5.0,2.0,Furnished,27000.0,29.24,Rent,"2 BHK Flat for Rent in Paranjape Broadway, Wak..."
3,Pune,Wakad,Flat,3,923.5,8.0,3.0,Semi-Furnished,42000.0,45.48,Rent,"3 BHK Flat for Rent in Paramount Altissimo, Wa..."
4,Pune,Wakad,Flat,2,923.5,11.0,2.0,Furnished,38000.0,41.15,Rent,"2 BHK Flat for Rent in Wakad, Pune"
